# Verifica dell'infrastruttura
Controlliamo Python 3.12 e, su richiesta, una connessione di sola lettura a PostgreSQL. Nessun caricamento o benchmark.
Avviare Jupyter e soltanto PostgreSQL con Docker Compose; gli script PowerShell sono facoltativi. Le credenziali sono iniettate da Compose: non copiarle nelle celle.


In [1]:
import sys
import psycopg

assert sys.version_info[:2] == (3, 12), 'Il kernel deve usare Python 3.12'
print('Python:', sys.version.split()[0])
print('Driver psycopg:', psycopg.__version__)
VERIFICA_CONNESSIONE = True  # True solo con PostgreSQL attivo.


Python: 3.12.12
Driver psycopg: 3.2.10


## Connessione e query di controllo
Psycopg legge PGHOST, PGPORT, PGDATABASE, PGUSER e PGPASSWORD dall'ambiente del container. `postgres` è il nome DNS del servizio nella rete Compose; `localhost` indicherebbe invece il container Jupyter.
I blocchi `with` chiudono cursore e connessione anche in caso di errore. Il risultato atteso della prima colonna è 1.

In [2]:
if VERIFICA_CONNESSIONE is True:
    try:
        with psycopg.connect(connect_timeout=5) as connection:
            with connection.cursor() as cursor:
                cursor.execute('SELECT 1, current_database(), current_user, version()')
                check, database, user, version = cursor.fetchone()
                assert check == 1
                print('Connessione OK:', database, '| utente:', user)
                print(version)
    except psycopg.OperationalError as error:
        raise RuntimeError('PostgreSQL non raggiungibile: verificare Docker Compose, healthcheck e credenziali locali.') from error
else:
    print('Operazione disabilitata: VERIFICA_CONNESSIONE=False.')


Connessione OK: flights | utente: flights_lab
PostgreSQL 17.6 (Debian 17.6-2.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


## Persistenza e arresto
`docker compose --profile postgres stop postgres` arresta il servizio e conserva il volume; Jupyter resta attivo. Dopo il riavvio si può ripetere il controllo.
Questo verifica disponibilità, non prestazioni. Per la presentazione usare `05_demo.ipynb`; per i risultati salvati `06_results_analysis.ipynb`, anche senza database attivi.
